# Блок 1 — Подготовка данных

Загрузка, очистка и аугментация датасетов для обучения.

## Что происходит в этом блоке

1. Загружаем данные из 8+ источников (GoEmotions RU, CEDR, Izard, Dusha, BRIGHTER, Aniemore, RuReviews, RuSentiTweet)
2. Балансируем классы (MAX_PER_CLASS = 15k)
3. Аугментируем редкие классы (disgust, fear, anger) через rut5 + back-translation
4. Сохраняем на диск для блока 02

> **Предобработка**: только базовая очистка текста (URL, HTML, whitespace).
> Лемматизацию и стоп-слова не убираем — трансформеры обучены на живой морфологии
> и справляются с ней сами. Лемматизация снижает качество на 2-5% F1.

In [ ]:
import sys, os, gc

PROJECT_ROOT = '/kaggle/input/sentiment-analysis' if os.path.exists('/kaggle') else os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

WORKING_DIR = '/kaggle/working' if os.path.exists('/kaggle') else '../results'
os.makedirs(WORKING_DIR, exist_ok=True)

import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_from_disk, Dataset, DatasetDict
from sklearn.utils import resample

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'WORKING_DIR:  {WORKING_DIR}')

## 1. Конфигурация

In [ ]:
from src.data_loader import EKMAN_LABEL_NAMES, EKMAN_ID2LABEL

LABEL_NAMES = EKMAN_LABEL_NAMES
NUM_LABELS  = len(LABEL_NAMES)
SEED = 42

# ── Балансировка Stage-1 ───────────────────────────────────────────────────
# Желаемое число примеров на класс в train-сплите.
# Кап на полный пул вычисляется автоматически с учётом test_size и val_size.
TARGET_TRAIN_PER_CLASS = 15_000
_S1_TEST_SIZE = 0.15
_S1_VAL_SIZE  = 0.15
MAX_PER_CLASS = round(TARGET_TRAIN_PER_CLASS / ((1 - _S1_TEST_SIZE) * (1 - _S1_VAL_SIZE)))  # ≈ 20,761

# ── Пути ──────────────────────────────────────────────────────────────────
S1_DATA_PATH     = f'{WORKING_DIR}/stage1_data'
S1_AUG_DATA_PATH = f'{WORKING_DIR}/stage1_data_augmented'
S2_DATA_PATH     = f'{WORKING_DIR}/stage2_data'
S2_AUG_DATA_PATH = f'{WORKING_DIR}/stage2_data_augmented'

# ── Балансировка Stage-1 ───────────────────────────────────────────────────

# ── Аугментация ────────────────────────────────────────────────────────────
USE_AUGMENTATION  = True    # False — пропустить аугментацию (быстрее)
AUG_METHOD        = 'both'  # 'paraphrase' | 'backtranslation' | 'both'
AUG_TARGET_S1     = 3_000   # цель на класс в Stage-1
AUG_TARGET_S2     = 400     # цель на класс в Stage-2
AUG_N_VARIANTS    = 3       # парафразов на предложение (Stage-1)
AUG_N_VARIANTS_S2 = 5       # парафразов на предложение (Stage-2)
AUG_BATCH_SIZE    = 8

print('Конфигурация загружена')
print(f'  Stage-1 path:     {S1_DATA_PATH}')
print(f'  Stage-1 aug path: {S1_AUG_DATA_PATH}')
print(f'  Stage-2 path:     {S2_DATA_PATH}')
print(f'  Stage-2 aug path: {S2_AUG_DATA_PATH}')
print(f'  Аугментация:      {USE_AUGMENTATION} (метод={AUG_METHOD})')

## 2. Загрузка Stage-1 (большой смешанный корпус)

Основные источники:
- `ru_go_emotions` raw (~211k) — переводной, широкое покрытие 7 классов
- `cedr-m7` (~9.4k) — нативный RU, **7 классов включая disgust + neutral**
- `ru-izard-emotions` (~30k) — перевод RU Reddit, 7 классов
- `brighter_hf` — нативный RU, Toloka-аннотация (SemEval-2025 Task 11)
- `aniemore/resd` (~4.5k) — нативный RU, 7 классов, STT-транскрипты
- `xed_russian` (~17k) — Helsinki-NLP субтитры, 8 эмоций Plutchik → Ekman
- `SberDevices/Dusha` — нативный RU транскрипты (~50k, 4 класса)
- `rureviews` + `rusentitweet` — объём (sentiment → Ekman, грубый маппинг)

**Ручная разметка** rureviews / rusentitweet загружается автоматически из `/kaggle/input/` если загружена.  
Заменяет грубый маппинг и добавляет полную 7-классовую разметку.


In [ ]:
from src.data_loader import (
    load_ru_go_emotions, load_cedr, load_cedr_m7, load_ru_izard_emotions,
    load_brighter_hf, load_dusha, load_aniemore_resd,
    load_xed_russian, load_rureviews, load_rusentitweet, merge_datasets,
)
from src.preprocessor import preprocess_batch

if os.path.exists(S1_DATA_PATH):
    print('Загружаем готовый Stage-1 датасет с диска...')
    stage1_ds = load_from_disk(S1_DATA_PATH)
else:
    print('Собираем Stage-1 датасет из всех источников...')
    s1_sources = {}

    # ── Основные датасеты эмоций ──────────────────────────────────────────────
    for name, loader, kwargs in [
        ('ru_go_emotions_raw', load_ru_go_emotions,    {'config': 'raw'}),
        ('cedr_m7',           load_cedr_m7,            {}),   # CEDR +disgust +neutral
        ('ru_izard',          load_ru_izard_emotions,  {}),
    ]:
        try:
            s1_sources[name] = loader(**kwargs)
        except Exception as e:
            print(f'  WARNING: {name} failed: {e}')
            if name == 'ru_go_emotions_raw':
                try:
                    s1_sources['ru_go_emotions'] = load_ru_go_emotions(config='simplified')
                    print('  Fallback: loaded simplified GoEmotions')
                except Exception as e2:
                    print(f'  WARNING: GoEmotions simplified also failed: {e2}')
            elif name == 'cedr_m7':
                try:
                    s1_sources['cedr'] = load_cedr()
                    print('  Fallback: loaded base CEDR (5 classes)')
                except Exception as e2:
                    print(f'  WARNING: cedr fallback also failed: {e2}')

    # ── Дополнительные источники ──────────────────────────────────────────────
    for name, loader in [
        ('brighter_hf',  load_brighter_hf),
        ('aniemore',     load_aniemore_resd),
        ('xed_russian',  load_xed_russian),
        ('rureviews',    load_rureviews),
        ('rusentitweet', load_rusentitweet),
    ]:
        try:
            s1_sources[name] = loader()
        except Exception as e:
            print(f'  WARNING: {name} failed: {e}')

    # ── Dusha ─────────────────────────────────────────────────────────────────
    try:
        s1_sources['dusha'] = load_dusha()
        print('  Dusha: loaded')
    except Exception as e:
        print(f'  INFO: dusha not available ({e})')

    stage1_ds = merge_datasets(s1_sources, test_size=0.15, val_size=0.15, seed=SEED,
                            max_per_class=MAX_PER_CLASS)
    stage1_ds = stage1_ds.map(
        lambda b: preprocess_batch(b, clean=True), batched=True, batch_size=1000
    )
    stage1_ds.save_to_disk(S1_DATA_PATH)
    print(f'\nStage-1 сохранён: {S1_DATA_PATH}')

print('\nStage-1 splits:')
for split in stage1_ds:
    print(f'  {split:12s}: {len(stage1_ds[split]):,}')


## 2б. Ручная разметка (rureviews + rusentitweet)

Kaggle-датасеты читаются автоматически:
- `ru_reviews_ekman7_{train,val,test}.csv` из `/kaggle/input/datasets/inexyy/ru-reviews-tweets/rureviews/`
- `rusentitweet_ekman7_{train,val,test}.csv` из `/kaggle/input/datasets/inexyy/ru-reviews-tweets/rusentitweet/`

Столбец разметки: **`ekman_emotion`**.  
Для rusentitweet: `original_label == neutral` → принудительно `neutral` (перекрывает `ekman_emotion`).

> Если датасеты не подключены, лоадеры скачают оригиналы с GitHub и применят грубый `pos→joy / neg→sadness`.


In [ ]:
# Ручная разметка rureviews / rusentitweet загружается автоматически в load_rureviews()
# и load_rusentitweet() — отдельных действий здесь не требуется.
# Kaggle paths:
#   /kaggle/input/datasets/inexyy/ru-reviews-tweets/rureviews/
#   /kaggle/input/datasets/inexyy/ru-reviews-tweets/rusentitweet/


## 3. Предварительный анализ (до аугментации)
Посмотрим какие классы недопредставлены и насколько.

In [ ]:
EMOTION_COLORS = {
    'anger':'#e74c3c','disgust':'#8e44ad','fear':'#2c3e50',
    'joy':'#f39c12','sadness':'#3498db','surprise':'#1abc9c','neutral':'#95a5a6',
}

df = stage1_ds['train'].to_pandas()
counts = df['label'].value_counts().sort_index()
total = len(df)

print('Распределение классов в Stage-1 train (ДО аугментации):')
for lid, cnt in counts.items():
    bar = '█' * int(cnt / total * 40)
    flag = '  ⚠ мало данных' if cnt < 3000 else ''
    print(f'  {EKMAN_ID2LABEL[lid]:<12}: {cnt:>7,}  ({cnt/total*100:4.1f}%)  {bar}{flag}')

fig, ax = plt.subplots(figsize=(10, 3))
labels_plot = [EKMAN_ID2LABEL[i] for i in counts.index]
bars = ax.bar(labels_plot, counts.values, color=[EMOTION_COLORS[l] for l in labels_plot])
ax.axhline(AUG_TARGET_S1, color='red', linestyle='--', alpha=0.7, label=f'Цель аугментации ({AUG_TARGET_S1:,})')
for bar, cnt in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100, f'{cnt:,}',
            ha='center', va='bottom', fontsize=9)
ax.set_title('Stage-1: распределение ДО аугментации')
ax.legend()
plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/distribution_before_aug.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Аугментация Stage-1 (редкие классы)

Парафраз через `cointegrated/rut5-base-paraphraser` + обратный перевод `Helsinki-NLP RU→EN→RU`.
Аугментируются только классы ниже порога `AUG_TARGET_S1`.

> Ориентировочное время на T4: ~10–20 мин при `method='both'`.  
> Если нужно быстрее — установи `AUG_METHOD = 'backtranslation'` (~5 мин) или `USE_AUGMENTATION = False`.

In [ ]:
from src.augmentation import augment_rare_classes

if not USE_AUGMENTATION:
    print('Аугментация отключена (USE_AUGMENTATION=False). Используем оригинальный Stage-1.')
    stage1_aug_ds = stage1_ds

elif os.path.exists(S1_AUG_DATA_PATH):
    print('Загружаем готовый аугментированный Stage-1 с диска...')
    stage1_aug_ds = load_from_disk(S1_AUG_DATA_PATH)

else:
    print(f'Аугментация Stage-1 (метод={AUG_METHOD}, цель={AUG_TARGET_S1}/класс)...')
    stage1_aug_ds = augment_rare_classes(
        dataset=stage1_ds,
        label_names=LABEL_NAMES,
        target_per_class=AUG_TARGET_S1,
        method=AUG_METHOD,
        n_variants=AUG_N_VARIANTS,
        batch_size=AUG_BATCH_SIZE,
        seed=SEED,
    )
    stage1_aug_ds.save_to_disk(S1_AUG_DATA_PATH)
    print(f'Stage-1 augmented сохранён: {S1_AUG_DATA_PATH}')

# Визуализация до/после
df_before = stage1_ds['train'].to_pandas()
df_after  = stage1_aug_ds['train'].to_pandas()
c_before  = df_before['label'].value_counts().sort_index()
c_after   = df_after['label'].value_counts().sort_index()

if USE_AUGMENTATION:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, (counts, title) in zip(axes, [(c_before, 'До аугментации'), (c_after, 'После аугментации')]):
        labels = [EKMAN_ID2LABEL[i] for i in counts.index]
        ax.bar(labels, counts.values, color=[EMOTION_COLORS[l] for l in labels])
        ax.set_title(f'{title}\n(train: {counts.sum():,})')
        ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.savefig(f'{WORKING_DIR}/s1_augmentation.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\nСравнение Stage-1:')
    print(f'  {"Класс":<12}  {"До":>8}  {"После":>8}  {"Δ":>7}')
    print('  ' + '-' * 44)
    for lid in range(NUM_LABELS):
        before = c_before.get(lid, 0)
        after  = c_after.get(lid, 0)
        delta  = after - before
        flag   = ' ← добавлено' if delta > 0 else ''
        print(f'  {LABEL_NAMES[lid]:<12}  {before:>8,}  {after:>8,}  +{delta:>5,}{flag}')
else:
    labels = [EKMAN_ID2LABEL[i] for i in c_before.index]
    plt.figure(figsize=(9, 3))
    plt.bar(labels, c_before.values, color=[EMOTION_COLORS[l] for l in labels])
    plt.title(f'Stage-1 train: {c_before.sum():,} примеров')
    plt.tight_layout(); plt.show()

## 5. Загрузка Stage-2 (чистый нативный корпус)

In [ ]:
from src.data_loader import load_stage2_clean

if os.path.exists(S2_DATA_PATH):
    print('Загружаем готовый Stage-2 датасет с диска...')
    stage2_ds = load_from_disk(S2_DATA_PATH)
else:
    print('Собираем Stage-2 (чистый) датасет...')
    # cedr_m7 заменяет base cedr — те же тексты + disgust и neutral
    stage2_ds = load_stage2_clean(
        use_cedr=True, use_cedr_m7=True,
        use_brighter_hf=True, use_aniemore=True,
        seed=SEED,
    )
    stage2_ds = stage2_ds.map(
        lambda b: preprocess_batch(b, clean=True), batched=True, batch_size=500
    )
    stage2_ds.save_to_disk(S2_DATA_PATH)
    print(f'Stage-2 сохранён: {S2_DATA_PATH}')

print('\nStage-2 splits:')
for split in stage2_ds:
    print(f'  {split:12s}: {len(stage2_ds[split]):,}')

print('\nПокрытие классов в Stage-2 train:')
df2 = stage2_ds['train'].to_pandas()
total2 = len(df2)
for lid in range(NUM_LABELS):
    cnt = (df2['label'] == lid).sum()
    print(f'  {EKMAN_ID2LABEL[lid]:<12}: {cnt:>5,}  ({cnt/total2*100:.1f}%)')

## 6. Аугментация Stage-2

In [ ]:
if not USE_AUGMENTATION:
    print('Аугментация отключена. Используем оригинальный Stage-2.')
    stage2_aug_ds = stage2_ds

elif os.path.exists(S2_AUG_DATA_PATH):
    print('Загружаем готовый аугментированный Stage-2 с диска...')
    stage2_aug_ds = load_from_disk(S2_AUG_DATA_PATH)

else:
    print(f'Аугментация Stage-2 (метод={AUG_METHOD}, цель={AUG_TARGET_S2}/класс)...')
    stage2_aug_ds = augment_rare_classes(
        dataset=stage2_ds,
        label_names=LABEL_NAMES,
        target_per_class=AUG_TARGET_S2,
        method=AUG_METHOD,
        n_variants=AUG_N_VARIANTS_S2,
        batch_size=AUG_BATCH_SIZE,
        seed=SEED,
    )
    stage2_aug_ds.save_to_disk(S2_AUG_DATA_PATH)
    print(f'Stage-2 augmented сохранён: {S2_AUG_DATA_PATH}')

print('\nStage-2 (финальный для обучения):')
df2a = stage2_aug_ds['train'].to_pandas()
total2a = len(df2a)
for lid in range(NUM_LABELS):
    cnt = (df2a['label'] == lid).sum()
    flag = '' if cnt >= AUG_TARGET_S2 else '  ← мало данных!'
    print(f'  {EKMAN_ID2LABEL[lid]:<12}: {cnt:>5,}  ({cnt/total2a*100:.1f}%){flag}')

## Итог

Данные подготовлены и сохранены на диск.

| Датасет | Путь | Описание |
|---|---|---|
| Stage-1 raw | `stage1_data/` | До аугментации |
| Stage-1 aug | `stage1_data_augmented/` | После аугментации |
| Stage-2 raw | `stage2_data/` | Чистый нативный RU |
| Stage-2 aug | `stage2_data_augmented/` | После аугментации |

**Следующий шаг:** запустить `02_training.ipynb`